# 03 · Feature engineering and leakage-safe splitting

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "carprice").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
from carprice.config import load_settings
S = load_settings()
P = S.paths
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.float_format", "{:,.3f}".format)
def show_fig(name):
    display(Image(filename=str(P.figures / name), width=900))
def md(text): display(Markdown(text))


## Features and the reasoning behind each
| feature | how it is computed | why | available for a new car? |
|---|---|---|---|
| `age` | reference_year − year | depreciation is the dominant effect | yes |
| `kms_per_year` | km / max(age,1) | usage intensity separates high-use from low-use cars of the same age | yes |
| `log_kms` | log1p(km) | diminishing effect of odometer | yes |
| `age_band` | 0-2, 3-5, 6-8, 9-12, 13+ | non-linear depreciation steps | yes |
| `company`, `model_family` | canonical brand; brand-aware model parsing | price level is brand/model specific | yes |
| `is_premium_brand` | brand ∈ luxury set | luxury cars have a different price scale | yes |
| `engine_l` | parsed from listing text (e.g. `1.2`) – **nullable** | only ~¼ of listings state it | yes (optional) |
| `transmission_hint` | `automatic`/`manual`/`unknown` from text only | absence ≠ manual, so `unknown` is a level | yes (optional) |
| brand / model price level | `TargetEncoder`, cross-fitted inside training folds | market-relative level without leaking a row's own price | learned in-pipeline |

**Rejected:** price-per-km (uses the target), brand mean price computed on the full data (target leakage), any feature from the test split.

In [2]:
from carprice.features.engineering import FeatureBuilder
tr = pd.read_csv(P.data_processed / "train.csv")
X = tr[["name","company","year","kms_driven","fuel_type"]]
fb = FeatureBuilder(json.load(open(P.metrics/"data_meta.json"))["reference_year"]).fit(X)
display(pd.concat([X.head(8).reset_index(drop=True), fb.transform(X.head(8)).drop(columns=["company","fuel_type","kms_driven"]).reset_index(drop=True)], axis=1))

,name,company,year,kms_driven,fuel_type,model_family,age,log_kms,kms_per_year,age_band,is_premium_brand,engine_l,transmission_hint
0,Hyundai Santro Xing XO eRLX Euro III,Hyundai,2007,"45,000.000",Petrol,Santro Xing,12.000,10.714,"3,750.000",9-12y,0.000,NaN,unknown
1,Mahindra Jeep CL550 MDI,Mahindra,2006,NaN,Diesel,Jeep,13.000,NaN,NaN,13y+,0.000,NaN,unknown
2,Hyundai Grand i10 Magna 1.2 Kappa VTVT,Hyundai,2014,"28,000.000",Petrol,Grand i10,5.000,10.240,"5,600.000",3-5y,0.000,1.200,unknown
3,Ford EcoSport Titanium 1.5L TDCi,Ford,2014,"36,000.000",Diesel,EcoSport,5.000,10.491,"7,200.000",3-5y,0.000,1.500,unknown
4,Ford Figo,Ford,2012,"41,000.000",Diesel,Figo,7.000,10.621,"5,857.143",6-8y,0.000,NaN,unknown
5,Hyundai Eon,Hyundai,2013,"25,000.000",Petrol,Eon,6.000,10.127,"4,166.667",6-8y,0.000,NaN,unknown
6,Ford EcoSport Ambiente 1.5L TDCi,Ford,2016,"24,530.000",Diesel,EcoSport,3.000,10.108,"8,176.667",3-5y,0.000,1.500,unknown
7,Maruti Suzuki Alto K10 VXi AMT,Maruti,2015,"60,000.000",Petrol,Alto K10,4.000,11.002,"15,000.000",3-5y,0.000,NaN,automatic


## Feature-set ablation (training data, grouped repeated CV)
`base` = what the original project used. Variants are compared with two probe models; ties go to the simpler set.

In [3]:
ab = json.load(open(P.metrics / "ablation.json"))
v = pd.DataFrame(ab["variant_table"]); display(v.pivot(index="variant", columns="model", values="mae_inr"))
md("Average rank (lower is better): " + ", ".join(f"`{k}` {x:.1f}" for k, x in ab["average_rank"].items()) + f" → probe winner **{ab['chosen_variant']}**.")
md("**Important finding:** Ridge prefers one-hot features, HistGradientBoosting prefers target encoding, so a single global feature set would bias the comparison. "
   "Every model is therefore evaluated on both `base_plus_parsed` and `target_encoded`.")

model,hist_gradient_boosting,ridge
variant,,
base,"150,245.617","128,516.467"
base_plus_parsed,"127,530.799","107,814.289"
target_encoded,"107,119.384","124,459.264"


Average rank (lower is better): `base` 3.0, `base_plus_parsed` 1.5, `target_encoded` 1.5 → probe winner **base_plus_parsed**.

**Important finding:** Ridge prefers one-hot features, HistGradientBoosting prefers target encoding, so a single global feature set would bias the comparison. Every model is therefore evaluated on both `base_plus_parsed` and `target_encoded`.

### Scaler choice (evidence-based)

In [4]:
display(pd.DataFrame(ab["scaler_table"]).pivot(index="model", columns="scaler", values="mae_inr"))
md("Chosen scalers (default kept unless >3% lower MAE): `" + json.dumps(ab["scaler_choice"]) + "`")

scaler,robust,standard
model,,
elasticnet,"111,051.001","110,945.278"
huber,"97,341.341","98,520.677"
knn,"135,334.619","130,581.710"
lasso,"114,860.656","114,991.087"
linear,"100,512.559","100,512.559"
mlp,"5,013,013.226","153,680.866"
ridge,"107,814.289","107,310.318"
svr,"103,004.952","102,317.861"


Chosen scalers (default kept unless >3% lower MAE): `{"elasticnet": "robust", "huber": "robust", "knn": "standard", "lasso": "robust", "linear": "robust", "mlp": "standard", "ridge": "robust", "svr": "robust"}`

## Leakage-safe splitting

In [5]:
from carprice.features.split import group_key, cv_splits
info = json.load(open(P.data_processed / "split.json"))
clean = pd.read_csv(P.data_interim / "cleaned.csv")
g = group_key(clean)
tr_g = set(g[clean.source_row.isin(info["train_rows"])]); te_g = set(g[clean.source_row.isin(info["test_rows"])])
assert not tr_g & te_g
md(f"Train {info['n_train']} / test {info['n_test']}; split hash `{info['split_hash']}`. **{len(tr_g & te_g)} listing groups straddle train/test** "
   f"(a 'group' = same model text + year + odometer±100 km, i.e. the same car listed twice).")
tr_df = clean[clean.source_row.isin(info["train_rows"])].reset_index(drop=True)
rows = []
for i, (a, b) in enumerate(cv_splits(tr_df, 5, 1, 5, 42)):
    rows.append({"fold": i, "n_train": len(a), "n_val": len(b), "median_price_val": tr_df.price_inr.iloc[b].median(), "pct_over_10L_val": (tr_df.price_inr.iloc[b] > 1e6).mean()*100})
display(pd.DataFrame(rows))
md("Folds are stratified by price quantile, so the expensive tail appears in every fold. **No listing dates exist**, so chronological validation was not possible (documented limitation).")

Train 581 / test 145; split hash `d6ffa1c84ed6422e`. **0 listing groups straddle train/test** (a 'group' = same model text + year + odometer±100 km, i.e. the same car listed twice).

,fold,n_train,n_val,median_price_val,pct_over_10L_val
0,0,464,117,"280,000.000",3.419
1,1,464,117,"299,999.000",5.128
2,2,464,117,"289,999.000",3.419
3,3,466,115,"280,000.000",7.826
4,4,466,115,"299,999.000",9.565


Folds are stratified by price quantile, so the expensive tail appears in every fold. **No listing dates exist**, so chronological validation was not possible (documented limitation).